# Wczytanie danych

Celem tego etapu jest przygotowanie danych do dalszej analizy. Sprawdzamy:

- czy pliki surowe wczytały się poprawnie,

- czy mają spójną strukturę,
- ile rekordów i kolumn zawiera każdy plik,
- oraz czy występują powtarzające się ogłoszenia między miesiącami.

In [42]:
import pandas as pd
from pathlib import Path

source_dir = Path("../Dane/Surowe")
files = sorted(source_dir.glob("apartments_pl_*.csv"))
files = [f.name for f in files]

In [43]:
df = pd.read_csv(source_dir / "apartments_pl_2023_08.csv")

print(f"Pierwszy plik: apartments_pl_2023_08.csv\nRozmiar: {df.shape[0]} wierszy, {df.shape[1]} kolumn")
df.head()

Pierwszy plik: apartments_pl_2023_08.csv
Rozmiar: 18905 wierszy, 28 kolumn


,id,city,type,squareMeters,rooms,floor,floorCount,buildYear,latitude,longitude,...,pharmacyDistance,ownership,buildingMaterial,condition,hasParkingSpace,hasBalcony,hasElevator,hasSecurity,hasStorageRoom,price
0,f8524536d4b09a0c8ccc0197ec9d7bde,szczecin,blockOfFlats,63.00,3.0,4.0,10.0,1980.0,53.378933,14.625296,...,0.413,condominium,concreteSlab,NaN,yes,yes,yes,no,yes,415000
1,accbe77d4b360fea9735f138a50608dd,szczecin,blockOfFlats,36.00,2.0,8.0,10.0,NaN,53.442692,14.559690,...,0.205,cooperative,concreteSlab,NaN,no,yes,yes,no,yes,395995
2,8373aa373dbc3fe7ca3b7434166b8766,szczecin,tenement,73.02,3.0,2.0,3.0,NaN,53.452222,14.553333,...,0.280,condominium,brick,NaN,no,no,no,no,no,565000
3,0a68cd14c44ec5140143ece75d739535,szczecin,tenement,87.60,3.0,2.0,3.0,NaN,53.435100,14.532900,...,0.087,condominium,brick,NaN,yes,yes,no,no,yes,640000
4,f66320e153c2441edc0fe293b54c8aeb,szczecin,blockOfFlats,66.00,3.0,1.0,3.0,NaN,53.410278,14.503611,...,0.514,condominium,NaN,NaN,no,no,no,no,no,759000


In [44]:
df.columns

Index(['id', 'city', 'type', 'squareMeters', 'rooms', 'floor', 'floorCount',
       'buildYear', 'latitude', 'longitude', 'centreDistance', 'poiCount',
       'schoolDistance', 'clinicDistance', 'postOfficeDistance',
       'kindergartenDistance', 'restaurantDistance', 'collegeDistance',
       'pharmacyDistance', 'ownership', 'buildingMaterial', 'condition',
       'hasParkingSpace', 'hasBalcony', 'hasElevator', 'hasSecurity',
       'hasStorageRoom', 'price'],
      dtype='str')

In [45]:
df.shape

(18905, 28)

In [46]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 18905 entries, 0 to 18904
Data columns (total 28 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id                    18905 non-null  str    
 1   city                  18905 non-null  str    
 2   type                  14866 non-null  str    
 3   squareMeters          18905 non-null  float64
 4   rooms                 18905 non-null  float64
 5   floor                 15467 non-null  float64
 6   floorCount            18643 non-null  float64
 7   buildYear             15634 non-null  float64
 8   latitude              18905 non-null  float64
 9   longitude             18905 non-null  float64
 10  centreDistance        18905 non-null  float64
 11  poiCount              18905 non-null  float64
 12  schoolDistance        18891 non-null  float64
 13  clinicDistance        18817 non-null  float64
 14  postOfficeDistance    18880 non-null  float64
 15  kindergartenDistance  18892 no

Sprawdzamy, czy wszystkie pliki sprzedażowe mają taki sam układ.

In [47]:
file_stats = []
for file in files:
    df_temp = pd.read_csv(source_dir / file)
    file_stats.append({
        "plik": file,
        "wiersze": len(df_temp),
        "kolumny": len(df_temp.columns),
        "pierwsza_kolumna": df_temp.columns[0],
        "ostatnia_kolumna": df_temp.columns[-1]
    })

file_stats = pd.DataFrame(file_stats)
file_stats['czy_spojne'] = file_stats['kolumny'] == file_stats['kolumny'].iloc[0]
file_stats

,plik,wiersze,kolumny,pierwsza_kolumna,ostatnia_kolumna,czy_spojne
0,apartments_pl_2023_08.csv,18905,28,id,price,True
1,apartments_pl_2023_09.csv,16997,28,id,price,True
2,apartments_pl_2023_10.csv,16690,28,id,price,True
3,apartments_pl_2023_11.csv,16302,28,id,price,True
4,apartments_pl_2023_12.csv,16483,28,id,price,True
5,apartments_pl_2024_01.csv,15521,28,id,price,True
6,apartments_pl_2024_02.csv,16361,28,id,price,True
7,apartments_pl_2024_03.csv,17318,28,id,price,True
8,apartments_pl_2024_04.csv,19259,28,id,price,True
9,apartments_pl_2024_05.csv,20231,28,id,price,True


Przechodzimy do połączenia 11 miesięcy sprzedaży w jeden zbiór.

In [48]:
sales_list = []

for file in files:
    df_temp = pd.read_csv(source_dir / file)
    
    month = file.replace("apartments_pl_", "").replace(".csv", "")
    df_temp["month"] = month
    
    sales_list.append(df_temp)

sales = pd.concat(sales_list, ignore_index=True)

Sprawdzamy liczbę wierszy i kolumn

In [49]:
sales.shape

(195568, 29)

In [50]:
sales.head()

,id,city,type,squareMeters,rooms,floor,floorCount,buildYear,latitude,longitude,...,ownership,buildingMaterial,condition,hasParkingSpace,hasBalcony,hasElevator,hasSecurity,hasStorageRoom,price,month
0,f8524536d4b09a0c8ccc0197ec9d7bde,szczecin,blockOfFlats,63.00,3.0,4.0,10.0,1980.0,53.378933,14.625296,...,condominium,concreteSlab,NaN,yes,yes,yes,no,yes,415000,2023_08
1,accbe77d4b360fea9735f138a50608dd,szczecin,blockOfFlats,36.00,2.0,8.0,10.0,NaN,53.442692,14.559690,...,cooperative,concreteSlab,NaN,no,yes,yes,no,yes,395995,2023_08
2,8373aa373dbc3fe7ca3b7434166b8766,szczecin,tenement,73.02,3.0,2.0,3.0,NaN,53.452222,14.553333,...,condominium,brick,NaN,no,no,no,no,no,565000,2023_08
3,0a68cd14c44ec5140143ece75d739535,szczecin,tenement,87.60,3.0,2.0,3.0,NaN,53.435100,14.532900,...,condominium,brick,NaN,yes,yes,no,no,yes,640000,2023_08
4,f66320e153c2441edc0fe293b54c8aeb,szczecin,blockOfFlats,66.00,3.0,1.0,3.0,NaN,53.410278,14.503611,...,condominium,NaN,NaN,no,no,no,no,no,759000,2023_08


In [51]:
sales["month"].value_counts().sort_index()

month
2023_08    18905
2023_09    16997
2023_10    16690
2023_11    16302
2023_12    16483
2024_01    15521
2024_02    16361
2024_03    17318
2024_04    19259
2024_05    20231
2024_06    21501
Name: count, dtype: int64

Upewniamy się, że typy danych to nadal liczby

In [52]:
sales.info()

<class 'pandas.DataFrame'>
RangeIndex: 195568 entries, 0 to 195567
Data columns (total 29 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   id                    195568 non-null  str    
 1   city                  195568 non-null  str    
 2   type                  153307 non-null  str    
 3   squareMeters          195568 non-null  float64
 4   rooms                 195568 non-null  float64
 5   floor                 160974 non-null  float64
 6   floorCount            193185 non-null  float64
 7   buildYear             163352 non-null  float64
 8   latitude              195568 non-null  float64
 9   longitude             195568 non-null  float64
 10  centreDistance        195568 non-null  float64
 11  poiCount              195568 non-null  float64
 12  schoolDistance        195400 non-null  float64
 13  clinicDistance        194840 non-null  float64
 14  postOfficeDistance    195320 non-null  float64
 15  kindergarte

Sprawdzamy duplikaty (Jeżeli mieszkanie było wystawione kilka miesięcy z rzędu to jego id może się pojawić kilka razy)

In [53]:
duplicate_count = sales['id'].duplicated().sum()
unique_ids = sales['id'].nunique()
total_rows = len(sales)
print(f"Liczba powtarzających się wpisów: {duplicate_count:,}")
print(f"Liczba unikalnych identyfikatorów: {unique_ids:,}")
print(f"Udział powtórzeń: {duplicate_count/total_rows*100:.2f}%")

Liczba powtarzających się wpisów: 102,601
Liczba unikalnych identyfikatorów: 92,967
Udział powtórzeń: 52.46%


Sprawdzamy liczbę mieszkań

In [54]:
sales["id"].nunique()


92967

Sprawdzamy liczbę wszystkich wierszy

In [55]:
sales.shape[0]

195568

Wniosek: część ofert się powtarza w kolejnych miesiącach

Zkontrolujmy duplikaty w danych

In [56]:
duplicate_ids = sales[sales["id"].duplicated()]["id"].head(1)

duplicate_ids

18905    71de9eb8a92ac12d7c94315228b70cfb
Name: id, dtype: str

In [57]:
sales[sales["id"] == duplicate_ids.iloc[0]]

,id,city,type,squareMeters,rooms,floor,floorCount,buildYear,latitude,longitude,...,ownership,buildingMaterial,condition,hasParkingSpace,hasBalcony,hasElevator,hasSecurity,hasStorageRoom,price,month
282,71de9eb8a92ac12d7c94315228b70cfb,szczecin,tenement,123.45,4.0,2.0,3.0,NaN,53.432586,14.535685,...,condominium,brick,NaN,no,yes,no,no,yes,799000,2023_08
18905,71de9eb8a92ac12d7c94315228b70cfb,szczecin,tenement,123.45,4.0,2.0,3.0,NaN,53.432586,14.535685,...,condominium,brick,NaN,no,yes,no,no,yes,799000,2023_09
36204,71de9eb8a92ac12d7c94315228b70cfb,szczecin,tenement,123.45,4.0,2.0,3.0,NaN,53.432586,14.535685,...,condominium,brick,NaN,no,yes,no,no,yes,799000,2023_10
52944,71de9eb8a92ac12d7c94315228b70cfb,szczecin,tenement,123.45,4.0,2.0,3.0,NaN,53.432586,14.535685,...,condominium,brick,NaN,no,yes,no,no,yes,799000,2023_11


## Podsumowanie wczytywania danych

- Pliki źródłowe są spójne pod względem liczby kolumn i struktury.
- Połączenie 11 miesięcy pozwala uzyskać pełny zbiór sprzedaży z okresu 2023-08 do 2024-06.
- W zbiorze występują powtarzające się identyfikatory ofert, co jest oczekiwane przy wystawianiu tych samych mieszkań w kolejnych miesiącach.
- W kolejnych etapach warto uwzględnić te powtórzenia przy analizie trendów cenowych oraz przy przygotowaniu danych do modelowania.
